ver2 : pred log ??, xgb parm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
import scipy

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import BayesianRidge, HuberRegressor, Ridge, OrthogonalMatchingPursuit
from sklearn.ensemble import GradientBoostingRegressor
import xgboost
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import LogisticRegression


from sklearn.model_selection import KFold, cross_val_score

In [ ]:
from sklearn.metrics import make_scorer

def rmsle(y_true, y_pred):
    # ???? ? ???? 0?? ??
    log_true = np.nan_to_num(np.log(y_true + 1))
    log_pred = np.nan_to_num(np.log(y_pred + 1))
    
    # RMSLE ??
    output = np.sqrt(np.mean((log_true - log_pred)**2))
    
    return output

RMSLE = make_scorer(rmsle, greater_is_better=False)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
train_da = pd.read_csv('train_dataset.csv')

In [ ]:
train = pd.concat([train, train_da],axis=0)

In [ ]:
train

In [ ]:
#train = train.loc[~(train['store_sales(in millions)']>20)]

In [ ]:
train.loc[~(train['store_sales(in millions)']>20),'store_sales(in millions)'] = 20

In [ ]:
train['unit_sales(in millions)'] = train['unit_sales(in millions)'].replace(6.0, 5.0)
test['unit_sales(in millions)'] = test['unit_sales(in millions)'].replace(6.0, 5.0)

In [ ]:
train = train.round(0)

In [ ]:
train.columns

In [ ]:
train = train.drop(['id','recyclable_package','low_fat'], axis=1)
test = test.drop(['id','recyclable_package','low_fat'], axis=1)

In [ ]:
def get_values(value):
    return value.values.reshape(-1, 1)

In [ ]:
ord_cols = ['store_sales(in millions)','gross_weight']
def scale(train, test):
    for col in train.columns:
        if col in ord_cols:
            print(col)
            scaler = MinMaxScaler()
            train[col] = scaler.fit_transform(get_values(train[col]))
            test[col] = scaler.transform(get_values(test[col]))
    return train, test

In [ ]:
train_updated, test_updated = scale(train, test)

In [ ]:
train_updated

florist, prepared_food, video_store, coffee_bar, store_sqft
store_sales(in millions), unit_sales(in millions)


In [ ]:
nom_cols = ['unit_sales(in millions)','total_children', 'num_children_at_home', 'avg_cars_at home(approx).1',#'recyclable_package', 'low_fat',
            'units_per_case','store_sqft', 'coffee_bar', 'video_store', 'prepared_food','florist']

In [ ]:
nom_cols = [
 'total_children',
 'num_children_at_home',
 'avg_cars_at home(approx).1',
 'store_sqft',
 'coffee_bar',
 'video_store',
 'salad_bar',
 'florist',
]

In [ ]:
enc = OneHotEncoder(categories = 'auto', dtype = 'int32', drop = 'first',sparse=False)

In [ ]:
nom_train = enc.fit_transform(train_updated[nom_cols])
nom_test = enc.transform(test_updated[nom_cols])

In [ ]:
train_update = np.hstack((np.array(train_updated[ord_cols]), nom_train))
test_update = np.hstack((np.array(test_updated[ord_cols]), nom_test))

In [ ]:
train_work_sprs =scipy.sparse.hstack([scipy.sparse.coo_matrix(train_update).astype('float64')]).tocsr()
test_work_sprs =scipy.sparse.hstack([scipy.sparse.coo_matrix(test_update).astype('float64')]).tocsr()

In [ ]:
best_params = {'n_estimators': 100, # not imp
             'learning_rate': 0.9, # + or - bad
             'max_depth': 15, # - bad + bad
             'lambda': 0.0150, # + bad 
             'alpha': 1.0e-08, # - is little good
             'colsample_bytree': 0.7, # imp (adding randomness to the training process.)
             'min_child_weight': 0, # + bad
             'booster': 'gbtree',
             'grow_policy': 'depthwise', 
             'objective': 'reg:squaredlogerror',
             'eval_metric': 'rmsle',
              }

In [ ]:
model = XGBRegressor(**best_params,n_jobs = -1, random_state=42)

classifier = LogisticRegression(random_state=42)
scores = cross_val_score(classifier, train_work_sprs, train.cost, scoring=RMSLE, cv = 3, n_jobs = -1)
classifier.fit(train_work_sprs, train.cost)

# Predict classes for testing set
y_pred = classifier.predict(test_work_sprs)
-np.mean(scores)

In [ ]:
scores = cross_val_score(model, train_work_sprs, train.cost, scoring=RMSLE, cv = 3, n_jobs = -1)

In [ ]:
-np.mean(scores)

In [ ]:
samp_sub = pd.read_csv('sample_submission.csv')

In [ ]:
model.fit(train_work_sprs, train.cost)

In [ ]:
preds = model.predict(test_work_sprs)

In [ ]:
samp_sub['cost'] = (preds)
samp_sub.to_csv("submission.csv", index=False)